# Data Quality Check

In [14]:
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import json
import re
from tqdm.auto import tqdm
from IPython.display import display
import random
import unicodedata
import math
from scipy.stats import hypergeom
from sklearn.model_selection import StratifiedGroupKFold
from itertools import combinations

In [2]:
BASE = Path("/home/ftp_246/data_5/多模態2026_ig_top200")

METADATA_DIR = BASE / "Metadata_by_Influencer"
VIDEO_DIR = BASE / "Video_data_fullest/ig_vedios"
JSON_DIR = BASE / "Prompt_Description/Fullset_v20"

PROFILE_PATH = Path(
    "/home/yingtzudai/Multimodal_SPoD_Video/data/Influencer_Profile.csv"
)

# 檢查路徑
paths = {
    "Profile": PROFILE_PATH,
    "Metadata": METADATA_DIR,
    "MP4": VIDEO_DIR,
    "JSON": JSON_DIR
}

for name, path in paths.items():
    print(f"{name}: {path.exists()} | {path}")

if not all(path.exists() for path in paths.values()):
    raise FileNotFoundError("請先修正不存在的資料路徑")

# 載入網紅 Profile，保留所有原始欄位
df_profile = pd.read_csv(
    PROFILE_PATH,
    dtype={"ig_id": "string", "ig_strong_id": "string"}
)

df_profile["ig_id"] = df_profile["ig_id"].str.strip()

print("Profile rows:", len(df_profile))

Profile: True | /home/yingtzudai/Multimodal_SPoD_Video/data/Influencer_Profile.csv
Metadata: True | /home/ftp_246/data_5/多模態2026_ig_top200/Metadata_by_Influencer
MP4: True | /home/ftp_246/data_5/多模態2026_ig_top200/Video_data_fullest/ig_vedios
JSON: True | /home/ftp_246/data_5/多模態2026_ig_top200/Prompt_Description/Fullset_v20
Profile rows: 203


## 讀取所有 Metadata，產生預期影片檔名

In [3]:
def format_modified_time(value):
    """將 modified_time_tw 轉為 YYYYMMDDHHMMSS。"""
    if pd.isna(value):
        return None

    s = str(value).strip()

    # 已經是 14 位數字
    if re.fullmatch(r"\d{14}", s):
        return s

    # 一般日期時間格式
    dt = pd.to_datetime(s, errors="coerce")

    if pd.notna(dt):
        return dt.strftime("%Y%m%d%H%M%S")

    return None


metadata_frames = []
metadata_errors = []

for csv_path in METADATA_DIR.rglob("*.csv"):
    try:
        df = pd.read_csv(
            csv_path,
            dtype={
                "id": "string",
                "media_id": "string",
                "post_owner.username": "string"
            },
            low_memory=False
        )

        required = [
            "id",
            "media_id",
            "post_owner.username",
            "modified_time_tw"
        ]

        missing = [c for c in required if c not in df.columns]

        if missing:
            metadata_errors.append({
                "file": str(csv_path),
                "error": f"Missing columns: {missing}"
            })
            continue

        df["metadata_path"] = str(csv_path)
        metadata_frames.append(df)

    except Exception as e:
        metadata_errors.append({
            "file": str(csv_path),
            "error": str(e)
        })

if not metadata_frames:
    raise ValueError("沒有成功讀取任何 Metadata CSV")

df_metadata = pd.concat(
    metadata_frames, ignore_index=True
)

# 統一欄位名稱
df_metadata = df_metadata.rename(columns={
    "id": "post_id",
    "post_owner.username": "ig_id"
})

for col in ["ig_id", "post_id", "media_id"]:
    df_metadata[col] = df_metadata[col].astype("string").str.strip()

df_metadata["modified_timestamp"] = (
    df_metadata["modified_time_tw"]
    .apply(format_modified_time)
)

# 確保組合檔名所需的欄位完整
valid_name = (
    df_metadata["ig_id"].notna()
    & df_metadata["post_id"].notna()
    & df_metadata["media_id"].notna()
    & df_metadata["modified_timestamp"].notna()
)

df_metadata["expected_video_filename"] = pd.NA

df_metadata.loc[valid_name, "expected_video_filename"] = (
    df_metadata.loc[valid_name, "ig_id"]
    + "-"
    + df_metadata.loc[valid_name, "modified_timestamp"]
    + "-"
    + df_metadata.loc[valid_name, "media_id"]
    + ".mp4"
)

print("Metadata rows:", len(df_metadata))
print("CSV read errors:", len(metadata_errors))
print("Rows with valid filenames:", valid_name.sum())

display(
    df_metadata[
        [
            "ig_id",
            "post_id",
            "media_id",
            "modified_time_tw",
            "expected_video_filename"
        ]
    ].head()
)

Metadata rows: 19443
CSV read errors: 0
Rows with valid filenames: 19443


,ig_id,post_id,media_id,modified_time_tw,expected_video_filename
0,traveggo,916191564143111,916191564143111,2025-08-05 00:37:18+08:00,traveggo-20250805003718-916191564143111.mp4
1,traveggo,3243955289117081,3243955289117081,2025-07-19 18:17:49+08:00,traveggo-20250719181749-3243955289117081.mp4
2,traveggo,1604802410544990,1604802410544990,2025-11-19 06:46:44+08:00,traveggo-20251119064644-1604802410544990.mp4
3,traveggo,849685401437704,849685401437704,2025-02-17 01:26:48+08:00,traveggo-20250217012648-849685401437704.mp4
4,traveggo,1452518233264809,1452518233264809,2025-09-17 14:44:07+08:00,traveggo-20250917144407-1452518233264809.mp4


## 掃描 MP4 與 JSON，建立 Post-level Inventory

In [4]:
# 1. 掃描所有 MP4
mp4_rows = []

for path in VIDEO_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() == ".mp4":
        mp4_rows.append({
            "ig_id": path.parent.name,
            "video_filename": path.name,
            "video_path": str(path),
            "video_size_mb": round(
                path.stat().st_size / (1024 ** 2), 2
            )
        })

df_mp4 = pd.DataFrame(
    mp4_rows,
    columns=[
        "ig_id", "video_filename",
        "video_path", "video_size_mb"
    ]
)

# 2. 掃描 JSON
json_rows = []
json_errors = []

for path in JSON_DIR.rglob("*.json"):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        video_name = data.get("videoName")

        if not isinstance(video_name, str):
            video_name = None

        json_rows.append({
            "ig_id": path.parent.name,
            "video_filename": (
                Path(video_name).name if video_name else None
            ),
            "json_path": str(path)
        })

    except (OSError, ValueError, TypeError) as e:
        json_errors.append({
            "file": str(path),
            "error": str(e)
        })

df_json = pd.DataFrame(
    json_rows,
    columns=["ig_id", "video_filename", "json_path"]
)

# 3. 以 ig_id + 完整檔名作為匹配鍵
keys = ["ig_id", "video_filename"]

# 找出重複鍵，避免 merge 造成多對多膨脹
mp4_key_counts = df_mp4.groupby(keys, dropna=False).size()
json_key_counts = df_json.groupby(keys, dropna=False).size()

mp4_unique = df_mp4[
    ~df_mp4.set_index(keys).index.isin(
        mp4_key_counts[mp4_key_counts > 1].index
    )
]

json_unique = df_json[
    ~df_json.set_index(keys).index.isin(
        json_key_counts[json_key_counts > 1].index
    )
]

# 4. 建立貼文清單
df_post_inventory = df_metadata.copy()

df_post_inventory["video_filename"] = (
    df_post_inventory["expected_video_filename"]
)

df_post_inventory = df_post_inventory.merge(
    mp4_unique,
    on=keys,
    how="left",
    validate="many_to_one"
)

df_post_inventory = df_post_inventory.merge(
    json_unique,
    on=keys,
    how="left",
    validate="many_to_one"
)

# 5. 建立完整度旗標
df_post_inventory["has_mp4"] = (
    df_post_inventory["video_path"].notna()
)

df_post_inventory["has_json"] = (
    df_post_inventory["json_path"].notna()
)

df_post_inventory["is_complete"] = (
    df_post_inventory["has_mp4"]
    & df_post_inventory["has_json"]
)

df_post_inventory["match_status"] = np.select(
    [
        df_post_inventory["is_complete"],
        df_post_inventory["has_mp4"],
        df_post_inventory["has_json"]
    ],
    [
        "complete",
        "mp4_only",
        "json_only"
    ],
    default="missing_both"
)

# 6. 重複貼文標記
df_post_inventory["duplicate_post_id"] = (
    df_post_inventory.duplicated(
        ["ig_id", "post_id"], keep=False
    )
)

# 簡化輸出欄位，保留研究所需資訊
post_columns = [
    "ig_id",
    "post_id",
    "media_id",
    "short_code",
    "text",
    "duration",
    "tags",
    "modified_time_tw",
    "video_filename",
    "video_path",
    "json_path",
    "video_size_mb",
    "has_mp4",
    "has_json",
    "is_complete",
    "match_status",
    "duplicate_post_id",
    "metadata_path"
]

df_post_inventory = df_post_inventory[
    [c for c in post_columns if c in df_post_inventory.columns]
]

print("Metadata rows:", len(df_post_inventory))
print("MP4 files:", len(df_mp4))
print("JSON records:", len(df_json))
print("Complete rows:", df_post_inventory["is_complete"].sum())

display(df_post_inventory.head())

Metadata rows: 19443
MP4 files: 19312
JSON records: 19116
Complete rows: 19102


,ig_id,post_id,media_id,short_code,text,duration,tags,modified_time_tw,video_filename,video_path,json_path,video_size_mb,has_mp4,has_json,is_complete,match_status,duplicate_post_id,metadata_path
0,traveggo,916191564143111,916191564143111,DE2RQZzy7ik,#同居觀察日記 Ep1\n同居是感情必修學分\n兩人住進同一個屋簷下\n首先迎來的 絕對是習...,90.02,{'0': '1833997590677795'},2025-08-05 00:37:18+08:00,traveggo-20250805003718-916191564143111.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,17.34,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
1,traveggo,3243955289117081,3243955289117081,DFK56DuvHOw,#同居觀察日記 Ep2\n經歷女生月經週期像換了4個女友……\n\n#情侶日常 #vegan...,31.81,NaN,2025-07-19 18:17:49+08:00,traveggo-20250719181749-3243955289117081.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,4.79,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
2,traveggo,1604802410544990,1604802410544990,DFNcyBZSSvJ,#同居觀察日記 Ep3\n我以為我在女生界洗澡算快了\n同居才發現男女洗澡速度真的差好多😂\...,89.20,{'0': '459340223188257'},2025-11-19 06:46:44+08:00,traveggo-20251119064644-1604802410544990.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,16.42,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
3,traveggo,849685401437704,849685401437704,DFXxOBDv1YN,帥哥帥妹祝大家新年快樂❤️\n2025 蛇麼都順利 平安 健康 心中有愛,15.16,NaN,2025-02-17 01:26:48+08:00,traveggo-20250217012648-849685401437704.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,2.81,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
4,traveggo,1452518233264809,1452518233264809,DGYHxkTvIr9,#同居觀察日記 Ep4 \n啊不是要保護我？？？？？？,40.87,NaN,2025-09-17 14:44:07+08:00,traveggo-20250917144407-1452518233264809.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,5.06,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...


## 產生網紅完整統計表

In [5]:
# 1. 以唯一 post_id 計算貼文數
post_stats = (
    df_post_inventory
    .groupby("ig_id", dropna=False)
    .agg(
        metadata_row_count=("post_id", "size"),
        metadata_post_count=("post_id", "nunique"),
        matched_video_count=("has_mp4", "sum"),
        matched_json_count=("has_json", "sum"),
        fully_matched_count=("is_complete", "sum"),
        duplicate_post_rows=("duplicate_post_id", "sum")
    )
    .reset_index()
)

# 2. MP4 實際檔案數
mp4_stats = (
    df_mp4.groupby("ig_id")
    .agg(
        mp4_count=("video_path", "size"),
        total_video_size_mb=("video_size_mb", "sum")
    )
    .reset_index()
)

# 3. JSON 實際檔案數
json_stats = (
    df_json.groupby("ig_id")
    .agg(json_count=("json_path", "size"))
    .reset_index()
)

# 4. 合併 Profile 與所有統計
# 使用 outer，避免遺漏 Profile 以外的帳號
df_influencer_summary = (
    df_profile
    .merge(
        post_stats,
        on="ig_id",
        how="outer",
        indicator="profile_merge_status"
    )
    .merge(mp4_stats, on="ig_id", how="outer")
    .merge(json_stats, on="ig_id", how="outer")
)

count_columns = [
    "metadata_row_count",
    "metadata_post_count",
    "matched_video_count",
    "matched_json_count",
    "fully_matched_count",
    "duplicate_post_rows",
    "mp4_count",
    "json_count"
]

for col in count_columns:
    df_influencer_summary[col] = (
        df_influencer_summary[col]
        .fillna(0)
        .astype(int)
    )

df_influencer_summary["total_video_size_mb"] = (
    df_influencer_summary["total_video_size_mb"]
    .fillna(0)
)

# 5. 計算完整率
denominator = (
    df_influencer_summary["metadata_post_count"]
    .replace(0, np.nan)
)

df_influencer_summary["complete_rate"] = (
    df_influencer_summary["fully_matched_count"]
    / denominator
)

# 6. 標記是否有可用資料
df_influencer_summary["has_usable_video"] = (
    df_influencer_summary["fully_matched_count"] > 0
)

# 7. 將重要欄位放在前面，其餘 Profile 欄位保留
priority_cols = [
    "ig_id",
    "ig_strong_id",
    "AsiaKOL_category",
    "Followers",
    "Following",
    "posts",
    "metadata_post_count",
    "mp4_count",
    "json_count",
    "matched_video_count",
    "matched_json_count",
    "fully_matched_count",
    "complete_rate",
    "has_usable_video",
    "total_video_size_mb"
]

priority_cols = [
    c for c in priority_cols
    if c in df_influencer_summary.columns
]

other_cols = [
    c for c in df_influencer_summary.columns
    if c not in priority_cols
]

df_influencer_summary = (
    df_influencer_summary[priority_cols + other_cols]
    .sort_values(
        "fully_matched_count",
        ascending=False
    )
    .reset_index(drop=True)
)

display(df_influencer_summary)

print("Total influencers:", len(df_influencer_summary))
print(
    "Influencers with complete videos:",
    df_influencer_summary["has_usable_video"].sum()
)
print(
    "Total complete post rows:",
    df_influencer_summary["fully_matched_count"].sum()
)

,ig_id,ig_strong_id,AsiaKOL_category,Followers,Following,posts,metadata_post_count,mp4_count,json_count,matched_video_count,...,has_usable_video,total_video_size_mb,Aisa_Order,person_name,ig_name,ig_url,AsiaKOL_followers,metadata_row_count,duplicate_post_rows,profile_merge_status
0,dr.acne_scar_queen,4833663962,"知識教育,醫療健康",189848,259,2943,639,639,639,639,...,True,4142.59,137,痘疤醫生---莊盈彥醫師,痘疤女王 莊盈彥,https://www.instagram.com/dr.acne_scar_queen/,"922,505",639,0,both
1,mayukichou,18346291,"時尚潮流,高階經理人",309258,1748,8861,538,538,538,538,...,True,2242.52,163,周小葳,葳老闆｜周品均,https://www.instagram.com/mayukichou/,"636,488",538,0,both
2,cocowine0205,2307480066,"美食料理,趣味搞笑",157798,782,1593,391,391,391,391,...,True,5889.88,98,可可酒精,可可酒精,https://www.instagram.com/cocowine0205/,"853,725",391,0,both
3,yga0721,45717781,趣味搞笑,1495461,501,8178,344,344,342,344,...,True,4807.44,1,蔡阿嘎,蔡阿嘎,https://www.instagram.com/yga0721/,"5,893,570",344,0,both
4,ladyflavor_cui,3656815405,NaN,369611,122,1378,342,342,342,342,...,True,4844.68,14,好味小姐,好味小姐脆脆,https://www.instagram.com/ladyflavor_cui/,NaN,342,0,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198,alvin701,1972535324,知識教育,187869,142,1329,1,0,0,0,...,False,0.00,115,艾爾文,艾爾文,https://www.instagram.com/alvin701/,"1,563,768",1,0,both
199,lynnwu0219_2,47642521277,帶貨分潤,76276,93,21,0,0,0,0,...,False,0.00,39,泱泱LYNN,泱泱LYNN,https://www.instagram.com/lynnwu0219_2/,"1,965,591",0,0,left_only
200,skauramomo,515352630,"時尚潮流,表演藝術",0,0,0,35,0,0,0,...,False,0.00,195,紫玥桃,紫玥桃 Cosplay,https://www.instagram.com/skauramomo/,"676,777",35,0,both
201,oldwangwang,3216171228,"趣味搞笑,時事討論",121681,57,289,1,1,1,0,...,False,5.13,177,老王-oldwang,老王 oldwong,https://www.instagram.com/oldwangwang/,"2,501,639",1,0,both


Total influencers: 203
Influencers with complete videos: 196
Total complete post rows: 19102


## 建立集中式 Data Quality Summary

In [6]:
# 重複貼文 ID
duplicate_posts = (
    df_post_inventory["duplicate_post_id"].sum()
)

# 沒有對應到 Metadata 的 MP4
expected_keys = df_metadata[
    ["ig_id", "expected_video_filename"]
].dropna().drop_duplicates().rename(
    columns={"expected_video_filename": "video_filename"}
)

orphan_mp4 = df_mp4.merge(
    expected_keys,
    on=["ig_id", "video_filename"],
    how="left",
    indicator=True
)

orphan_mp4_count = (
    orphan_mp4["_merge"] == "left_only"
).sum()

# 沒有對應到 Metadata 的 JSON
orphan_json = df_json.merge(
    expected_keys,
    on=["ig_id", "video_filename"],
    how="left",
    indicator=True
)

orphan_json_count = (
    orphan_json["_merge"] == "left_only"
).sum()

# Profile 沒有出現的網紅
profile_ids = set(df_profile["ig_id"].dropna())

all_data_ids = (
    set(df_metadata["ig_id"].dropna())
    | set(df_mp4["ig_id"].dropna())
    | set(df_json["ig_id"].dropna())
)

unknown_influencers = all_data_ids - profile_ids

# Profile 中重複的識別碼
duplicate_strong_ids = (
    df_profile["ig_strong_id"]
    .dropna()
    .duplicated()
    .sum()
)

quality_checks = [
    ("Metadata 總列數", len(df_metadata)),
    ("唯一網紅數（所有來源）", len(profile_ids | all_data_ids)),
    ("實際 MP4 數", len(df_mp4)),
    ("成功解析 JSON 數", len(df_json)),
    ("完整匹配貼文列數",
     int(df_post_inventory["is_complete"].sum())),
    ("缺少 MP4 的貼文列數",
     int((~df_post_inventory["has_mp4"]).sum())),
    ("缺少 JSON 的貼文列數",
     int((~df_post_inventory["has_json"]).sum())),
    ("缺少 MP4 與 JSON 的貼文列數",
     int((df_post_inventory["match_status"] == "missing_both").sum())),
    ("重複貼文 ID 列數", int(duplicate_posts)),
    ("重複 ig_strong_id 數", int(duplicate_strong_ids)),
    ("沒有 Metadata 對應的 MP4 數", int(orphan_mp4_count)),
    ("沒有 Metadata 對應的 JSON 數", int(orphan_json_count)),
    ("Profile 以外的網紅數", len(unknown_influencers)),
    ("Metadata CSV 讀取錯誤數", len(metadata_errors)),
    ("JSON 解析錯誤數", len(json_errors)),
    ("重複 MP4 匹配鍵數",
     int((mp4_key_counts > 1).sum())),
    ("重複 JSON 匹配鍵數",
     int((json_key_counts > 1).sum()))
]

df_data_quality = pd.DataFrame(
    quality_checks,
    columns=["check_item", "count"]
)

display(df_data_quality)

# 找出 Metadata 有紀錄，但找不到對應 MP4 的貼文
df_missing_videos = df_post_inventory.loc[
    ~df_post_inventory["has_mp4"]
].copy()

# 重新產生預期檔名，方便查找
df_missing_videos["expected_video_filename"] = (
    df_missing_videos["video_filename"]
)

# 選擇要顯示的欄位
columns = [
    "ig_id",
    "post_id",
    "media_id",
    "modified_time_tw",
    "expected_video_filename",
    "has_json",
    "match_status"
]

df_missing_videos = df_missing_videos[columns]

print(f"Missing MP4 posts: {len(df_missing_videos)}")

# 顯示前 20 筆缺少 MP4 的貼文
display(df_missing_videos.head(20))

,check_item,count
0,Metadata 總列數,19443
1,唯一網紅數（所有來源）,203
2,實際 MP4 數,19312
3,成功解析 JSON 數,19116
4,完整匹配貼文列數,19102
5,缺少 MP4 的貼文列數,133
6,缺少 JSON 的貼文列數,341
7,缺少 MP4 與 JSON 的貼文列數,133
8,重複貼文 ID 列數,0
9,重複 ig_strong_id 數,1


Missing MP4 posts: 133


,ig_id,post_id,media_id,modified_time_tw,expected_video_filename,has_json,match_status
65,traveggo,4648229675409463,4648229675409463,2025-09-06 22:44:10+08:00,traveggo-20250906224410-4648229675409463.mp4,False,missing_both
302,tzu_888,1215205530776275,1215205530776275,2025-12-09 15:31:21+08:00,tzu_888-20251209153121-1215205530776275.mp4,False,missing_both
1077,wia627,1852338622114597,1852338622114597,2025-09-20 04:58:47+08:00,wia627-20250920045847-1852338622114597.mp4,False,missing_both
1989,zxsdexz,1213717130801192,1213717130801192,2026-01-05 23:09:26+08:00,zxsdexz-20260105230926-1213717130801192.mp4,False,missing_both
1991,0_shufen,1165134668817062,1165134668817062,2025-11-23 04:00:36+08:00,0_shufen-20251123040036-1165134668817062.mp4,False,missing_both
2119,1_shiuan_0,1639657417044170,1639657417044170,2025-08-19 22:25:28+08:00,1_shiuan_0-20250819222528-1639657417044170.mp4,False,missing_both
2352,2uncle987,888601933921971,888601933921971,2025-11-22 01:00:34+08:00,2uncle987-20251122010034-888601933921971.mp4,False,missing_both
3289,1105ya,1051447247181222,1051447247181222,2025-12-12 13:43:16+08:00,1105ya-20251212134316-1051447247181222.mp4,False,missing_both
3501,ackj0301_ifbbpro,1646270259713007,1646270259713007,2026-01-16 03:33:22+08:00,ackj0301_ifbbpro-20260116033322-16462702597130...,False,missing_both
3598,ala780705,1258671062799202,1258671062799202,2025-01-01 14:33:38+08:00,ala780705-20250101143338-1258671062799202.mp4,False,missing_both


## 最終統計表

In [7]:
print("=== SPoD Dataset Overview ===")

print(
    "Total influencers:",
    df_influencer_summary["ig_id"].nunique()
)

print(
    "Influencers with complete data:",
    df_influencer_summary["has_usable_video"].sum()
)

print(
    "Total metadata posts:",
    df_influencer_summary["metadata_post_count"].sum()
)

print(
    "Total MP4 files:",
    df_influencer_summary["mp4_count"].sum()
)

print(
    "Total JSON files:",
    df_influencer_summary["json_count"].sum()
)

print(
    "Total fully matched rows:",
    df_influencer_summary["fully_matched_count"].sum()
)

print(
    "Total video storage (GB):",
    round(
        df_influencer_summary["total_video_size_mb"].sum() / 1024,
        2
    )
)

=== SPoD Dataset Overview ===
Total influencers: 203
Influencers with complete data: 196
Total metadata posts: 19443
Total MP4 files: 19312
Total JSON files: 19116
Total fully matched rows: 19102
Total video storage (GB): 235.73


# 抽樣與統計
**SPoD**：training : validating : testing = 7:1:2

**A simple formula for the calculation of sample size in pilot studies**: 樣本數n = ln(1−γ) / ln(1−p)。其中 p 是研究者在pilot study希望能抓到的「最低問題發生率」，γ 是信心水準。假設C = 0.95 p = 0.01，則 n = 299，也就是假設某種問題的發生率是0.01，且每篇post遇到問題的機會彼此獨立，抽查299篇就有至少95%的機會遇到一次這種問題

## 參數設定與資料檢查

In [11]:
SEED = 301
TARGET_RATIO = np.array([0.7, 0.1, 0.2])
SPLIT_NAMES = ["train", "validation", "test"]

PROJECT_DIR = Path(
    "/home/yingtzudai/Multimodal_SPoD_Video"
)
PROFILE_PATH = PROJECT_DIR / "data/Influencer_Profile.csv"
OUTPUT_DIR = PROJECT_DIR / "data/split"

df_profile = pd.read_csv(PROFILE_PATH, dtype={"ig_id": str})

# 沿用前面已經建立的完整配對清單
df = df_post_inventory.loc[
    df_post_inventory["is_complete"].eq(True)
].copy()

df["ig_id"] = df["ig_id"].astype("string").str.strip()
df["post_id"] = df["post_id"].astype("string").str.strip()

assert df["post_id"].notna().all()
assert not df.duplicated(["ig_id", "post_id"]).any(), \
    "存在重複貼文 ID，請先檢查"

def normalize_person(value):
    if pd.isna(value):
        return ""
    return re.sub(
        r"\s+", " ",
        unicodedata.normalize("NFKC", str(value))
    ).strip().casefold()

profile = df_profile[
    ["ig_id", "person_name",
     "AsiaKOL_category", "Followers", "Following"]
].copy()

profile["ig_id"] = profile["ig_id"].str.strip()

assert not profile["ig_id"].duplicated().any(), \
    "Profile 的 ig_id 不唯一"

# 避免重複的 Profile 欄位
df = df.drop(
    columns=[
        "person_name", "AsiaKOL_category",
        "Followers", "Following"
    ],
    errors="ignore"
)

df = df.merge(
    profile,
    on="ig_id",
    how="left",
    validate="many_to_one"
)

# 缺少 Profile 的帳號不得默默略過
missing_profile = df.loc[
    df["person_name"].isna(), "ig_id"
].drop_duplicates()

if len(missing_profile):
    raise ValueError(
        f"有 {len(missing_profile)} 個帳號缺少 Profile："
        f"{missing_profile.tolist()[:20]}"
    )

df["person_group"] = df["person_name"].apply(
    normalize_person
)

if df["person_group"].eq("").any():
    raise ValueError("存在空白 person_name，請先確認身分")

# 檢查多帳號人物
df_multi_accounts = (
    df[["person_name", "person_group", "ig_id"]]
    .drop_duplicates()
    .groupby(["person_name", "person_group"])["ig_id"]
    .agg(list)
    .reset_index()
)

df_multi_accounts["account_count"] = (
    df_multi_accounts["ig_id"].apply(len)
)

display(
    df_multi_accounts.loc[
        df_multi_accounts["account_count"] > 1
    ]
)

print("Complete posts:", len(df))
print("IG accounts:", df["ig_id"].nunique())
print("Unique persons:", df["person_group"].nunique())

,person_name,person_group,ig_id,account_count
82,好味小姐,好味小姐,"[ladyflavor_cui, ladyflavor_nowpo, ladyflavor]",3


Complete posts: 19102
IG accounts: 196
Unique persons: 194


## 讀取Endorsement Label 整理Category與粉絲規模

In [12]:
def normalize_label(value):
    if isinstance(value, (bool, np.bool_)):
        return "True" if value else "False"
    if isinstance(value, str):
        x = value.strip().lower()
        if x == "true":
            return "True"
        if x == "false":
            return "False"
    return "Unknown"


def read_endorsement(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        value = (
            obj.get("high_inference", {})
               .get("endorsement_analysis", {})
               .get("isEndorsement")
        )
        return normalize_label(value)
    except (OSError, ValueError, TypeError, AttributeError):
        return "Unknown"


# 直接從 JSON 讀取，確保每篇標籤來源一致
df["endorsement_label"] = [
    read_endorsement(path)
    for path in tqdm(
        df["json_path"],
        desc="Loading JSON labels",
        unit="post"
    )
]


def parse_categories(value):
    if pd.isna(value):
        return []
    return sorted(set(
        x.strip()
        for x in re.split(r"[,，]", str(value))
        if x.strip()
    ))


def parse_followers(value):
    if pd.isna(value):
        return np.nan
    x = re.sub(r"[,\s]", "", str(value))
    return pd.to_numeric(x, errors="coerce")


df["category_list"] = (
    df["AsiaKOL_category"].apply(parse_categories)
)

df["followers_num"] = (
    df["Followers"].apply(parse_followers)
)

# 先建立每個 IG 帳號的個別特徵
accounts = df[
    ["person_group", "ig_id",
     "category_list", "followers_num"]
].drop_duplicates(subset=["ig_id"])

# 聚合至「人物」層級
persons = (
    accounts.groupby("person_group")
    .agg(
        account_count=("ig_id", "nunique"),
        followers_max=("followers_num", "max")
    )
    .reset_index()
)

person_categories = (
    accounts.groupby("person_group")["category_list"]
    .agg(lambda groups: sorted(set(
        c for cats in groups for c in cats
    )))
    .reset_index(name="categories")
)

persons = persons.merge(
    person_categories, on="person_group"
)

# 粉絲規模：以有數值的人物建立四分位
persons["follower_group"] = "Unknown"

valid = persons["followers_max"].notna()

if valid.sum() >= 4:
    q = pd.qcut(
        persons.loc[valid, "followers_max"].rank(
            method="first"
        ),
        q=4,
        labels=["Q1", "Q2", "Q3", "Q4"]
    )
    persons.loc[valid, "follower_group"] = (
        q.astype(str).to_numpy()
    )

df = df.merge(
    persons[
        ["person_group", "follower_group"]
    ],
    on="person_group",
    how="left",
    validate="many_to_one"
)

display(df["endorsement_label"].value_counts())
display(persons["follower_group"].value_counts())

Loading JSON labels:   0%|          | 0/19102 [00:00<?, ?post/s]

endorsement_label
False    9956
True     9146
Name: count, dtype: int64

follower_group
Q1    49
Q4    49
Q3    48
Q2    48
Name: count, dtype: int64

## Grouped Stratified Split (7:1:2)
第一層使用 scikit-learn 的 StratifiedGroupKFold，讓同一人物的資料不能跨 Fold，同時盡量保留業配標籤分布。

第二層從 10 個 Fold 中選擇 7 個作 Train、1 個作 Validation、2 個作 Test，並比較多組候選分配，選出貼文比例、網紅類別與粉絲規模分布偏差較小的方案。

In [15]:
# 形成每個人物的特徵
person_features = persons.set_index("person_group").copy()

# 類別：人物層級 multi-hot
all_categories = sorted(set(
    c for cats in persons["categories"] for c in cats
))

for category in all_categories:
    person_features[f"cat::{category}"] = (
        person_features["categories"].apply(
            lambda x: int(category in x)
        )
    )

# 粉絲分組：人物層級 one-hot
follower_groups = ["Q1", "Q2", "Q3", "Q4", "Unknown"]

for group in follower_groups:
    person_features[f"follower::{group}"] = (
        person_features["follower_group"].eq(group).astype(int)
    )

person_features["person_count"] = 1

# Post 數與標籤數：貼文層級
post_counts = pd.crosstab(
    df["person_group"],
    df["endorsement_label"]
).reindex(
    columns=["True", "False", "Unknown"],
    fill_value=0
)

post_counts = post_counts.rename(
    columns={
        "True": "label::True",
        "False": "label::False",
        "Unknown": "label::Unknown"
    }
)

post_counts["post_count"] = post_counts.sum(axis=1)

person_features = person_features.join(post_counts)

feature_cols = (
    ["post_count", "person_count"]
    + ["label::True", "label::False", "label::Unknown"]
    + [f"cat::{c}" for c in all_categories]
    + [f"follower::{g}" for g in follower_groups]
)

person_features[feature_cols] = (
    person_features[feature_cols].fillna(0)
)

# 權重是本研究的明示設計選擇，而非文獻中的固定參數
WEIGHTS = {
    "size": 4.0,
    "labels": 2.0,
    "persons": 1.0,
    "categories": 1.0,
    "followers": 1.0
}

def split_score(split_matrix, total):
    """
    每列為 Train / Validation / Test。
    各特徵先換算為在三組中的分配比例，
    再衡量與 7:1:2 的絕對偏差。
    """
    def deviation(indices):
        if not indices:
            return 0.0

        actual = split_matrix[:, indices]
        totals = total[indices]
        usable = totals > 0

        if not usable.any():
            return 0.0

        shares = actual[:, usable] / totals[usable]
        return float(np.abs(
            shares - TARGET_RATIO[:, None]
        ).mean())

    col_index = {
        col: i for i, col in enumerate(feature_cols)
    }

    def idx(names):
        return [
            col_index[x] for x in names
            if x in col_index
        ]

    return (
        WEIGHTS["size"] * deviation(idx(["post_count"]))
        + WEIGHTS["labels"] * deviation(idx([
            "label::True", "label::False", "label::Unknown"
        ]))
        + WEIGHTS["persons"] * deviation(idx(["person_count"]))
        + WEIGHTS["categories"] * deviation(idx([
            f"cat::{c}" for c in all_categories
        ]))
        + WEIGHTS["followers"] * deviation(idx([
            f"follower::{g}" for g in follower_groups
        ]))
    )


# 多個可重現的候選分配，選擇分布偏差最小的一組
N_SEARCH_SEEDS = 20
best = None

for trial in tqdm(
    range(N_SEARCH_SEEDS),
    desc="Searching grouped splits",
    unit="trial"
):
    splitter = StratifiedGroupKFold(
        n_splits=10,
        shuffle=True,
        random_state=SEED + trial
    )

    # 每一篇 Post 對應到一個 Fold
    fold_ids = np.full(len(df), -1, dtype=int)

    for fold_id, (_, test_idx) in enumerate(
        splitter.split(
            X=np.zeros(len(df)),
            y=df["endorsement_label"],
            groups=df["person_group"]
        )
    ):
        fold_ids[test_idx] = fold_id

    # 將每個人物的 Fold 取得並確認唯一
    fold_df = pd.DataFrame({
        "person_group": df["person_group"].to_numpy(),
        "fold": fold_ids
    })

    assert (
        fold_df.groupby("person_group")["fold"]
        .nunique().max() == 1
    )

    person_folds = (
        fold_df.drop_duplicates("person_group")
        .set_index("person_group")["fold"]
    )

    feature_table = person_features[feature_cols].copy()
    feature_table["fold"] = person_folds

    fold_matrix = (
        feature_table.groupby("fold")[feature_cols]
        .sum()
        .reindex(range(10), fill_value=0)
        .to_numpy(dtype=float)
    )

    total = fold_matrix.sum(axis=0)

    # 10 Fold 中：7 Train / 1 Validation / 2 Test
    for val_fold in range(10):
        remaining_folds = [
            x for x in range(10) if x != val_fold
        ]

        for test_folds in combinations(remaining_folds, 2):
            train_folds = [
                x for x in remaining_folds
                if x not in test_folds
            ]

            split_matrix = np.stack([
                fold_matrix[train_folds].sum(axis=0),
                fold_matrix[val_fold],
                fold_matrix[list(test_folds)].sum(axis=0)
            ])

            score = split_score(split_matrix, total)

            if best is None or score < best["score"]:
                best = {
                    "score": score,
                    "seed": SEED + trial,
                    "val_fold": val_fold,
                    "test_folds": test_folds,
                    "person_folds": person_folds.copy()
                }


# 根據最佳結果建立最終 split
def assign_split(fold):
    if fold == best["val_fold"]:
        return "validation"
    if fold in best["test_folds"]:
        return "test"
    return "train"


person_split = (
    best["person_folds"]
    .map(assign_split)
    .rename("split")
)

df["split"] = df["person_group"].map(person_split)

print("Best random seed:", best["seed"])
print("Best distribution score:", round(best["score"], 5))

display(
    df["split"].value_counts().reindex(SPLIT_NAMES)
)

# 絕對不能讓同一人出現在多個資料集
assert df.groupby("person_group")["split"].nunique().max() == 1

# 也不能重複或遺漏原始貼文
assert len(df) == len(df_post_inventory.loc[
    df_post_inventory["is_complete"].eq(True)
])

assert df["split"].notna().all()

print("Group leakage check: PASSED")

Searching grouped splits:   0%|          | 0/20 [00:00<?, ?trial/s]

Best random seed: 309
Best distribution score: 0.13792


split
train         13408
validation     1910
test           3784
Name: count, dtype: int64

Group leakage check: PASSED


In [16]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_cols = [
    "person_group",
    "person_name",
    "ig_id",
    "post_id",
    "media_id",
    "text",
    "video_path",
    "json_path",
    "endorsement_label",
    "AsiaKOL_category",
    "Followers",
    "Following",
    "follower_group",
    "split"
]

export_cols = [c for c in export_cols if c in df.columns]

for name in tqdm(
    SPLIT_NAMES,
    desc="Saving splits",
    unit="dataset"
):
    subset = df.loc[df["split"] == name, export_cols]

    subset.to_csv(
        OUTPUT_DIR / f"{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"{name}: {len(subset):,} posts")

# 儲存人物與帳號切分對照
df_person_split = (
    df[["person_group", "person_name", "ig_id", "split"]]
    .drop_duplicates()
    .sort_values(["split", "person_group", "ig_id"])
)

df_person_split.to_csv(
    OUTPUT_DIR / "person_split_mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved to:", OUTPUT_DIR)

Saving splits:   0%|          | 0/3 [00:00<?, ?dataset/s]

train: 13,408 posts
validation: 1,910 posts
test: 3,784 posts
Saved to: /home/yingtzudai/Multimodal_SPoD_Video/data/split


## 從 Training Set 抽取 299 篇 Pilot Sample
這裡遵循 Viechtbauer 的計算方式，並保留簡單隨機抽樣的特性：
- 只從 train.csv 抽取。
- 每篇貼文有相同的入選機會。
- 不放回抽樣。
- 不強制 50:50 業配比例。
- 固定 Random Seed，方便重現。
- 抽樣完成後檢查網紅覆蓋度與標籤分布。

In [18]:
# 參照 Viechtbauer et al. (2015)
CONFIDENCE = 0.95
PROBLEM_RATE = 0.01
PILOT_SEED = 3

pilot_n = math.ceil(
    math.log(1 - CONFIDENCE)
    / math.log(1 - PROBLEM_RATE)
)

train_df = df.loc[df["split"] == "train"].copy()

if len(train_df) < pilot_n:
    raise ValueError(
        f"Training set 只有 {len(train_df)} 篇，"
        f"不足以抽出 {pilot_n} 篇"
    )

# 不放回 Simple Random Sampling
df_pilot = train_df.sample(
    n=pilot_n,
    replace=False,
    random_state=PILOT_SEED
).copy()

df_pilot["pilot_sample"] = True

# 核對 Pilot 只包含 Training 資料
assert df_pilot["split"].eq("train").all()

assert len(df_pilot) == pilot_n

assert not df_pilot.duplicated(
    ["ig_id", "post_id"]
).any()

# 檢查 Pilot 的所有貼文是否都來自 Train
train_keys = set(zip(
    train_df["ig_id"], train_df["post_id"]
))

pilot_keys = set(zip(
    df_pilot["ig_id"], df_pilot["post_id"]
))

assert pilot_keys.issubset(train_keys)

# 儲存
df_pilot[export_cols].to_csv(
    OUTPUT_DIR / f"pilot_{pilot_n}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Pilot sample size:", len(df_pilot))
print("Unique persons:", df_pilot["person_group"].nunique())
print("Unique IG accounts:", df_pilot["ig_id"].nunique())

print("\nPseudo-label distribution:")
display(
    df_pilot["endorsement_label"]
    .value_counts(dropna=False)
    .to_frame("posts")
)

print("\nPosts per person:")
display(
    df_pilot["person_group"]
    .value_counts()
    .describe()
)

Pilot sample size: 299
Unique persons: 97
Unique IG accounts: 98

Pseudo-label distribution:


,posts
endorsement_label,
False,163
True,136



Posts per person:


count    97.000000
mean      3.082474
std       2.925000
min       1.000000
25%       1.000000
50%       2.000000
75%       4.000000
max      19.000000
Name: count, dtype: float64

## 資料分布統計
- df_split_summary：Train / Validation / Test / Pilot 的總體統計。
- df_label_summary：各組的 True、False、Unknown 數量及比例。
- df_category_summary：不同網紅類型在各資料集的覆蓋情況。

In [20]:
# 將完整母體與不同資料集放在一起比較
datasets = {
    "full": df,
    "train": df.loc[df["split"] == "train"],
    "validation": df.loc[df["split"] == "validation"],
    "test": df.loc[df["split"] == "test"],
    "pilot": df_pilot
}

# 1. 整體統計
summary_rows = []

for name, data in tqdm(
    datasets.items(),
    desc="Building summary tables",
    total=len(datasets)
):
    n_posts = len(data)

    summary_rows.append({
        "dataset": name,
        "posts": n_posts,
        "post_ratio_%": round(
            n_posts / len(df) * 100, 2
        ),
        "persons": data["person_group"].nunique(),
        "ig_accounts": data["ig_id"].nunique(),
        "true": int(
            data["endorsement_label"].eq("True").sum()
        ),
        "false": int(
            data["endorsement_label"].eq("False").sum()
        ),
        "unknown": int(
            data["endorsement_label"].eq("Unknown").sum()
        ),
        "true_ratio_%": round(
            data["endorsement_label"].eq("True").mean()
            * 100, 2
        ) if n_posts else np.nan
    })

df_split_summary = pd.DataFrame(summary_rows)

display(df_split_summary)


# 2. Label 分布（貼文層級）
# df_label_summary = pd.crosstab(
#     pd.concat([
#         data.assign(dataset=name)
#         for name, data in datasets.items()
#     ])["dataset"],
#     pd.concat(datasets.values())["endorsement_label"]
#     .reset_index(drop=True)
# ).reindex(
#     index=list(datasets),
#     columns=["True", "False", "Unknown"],
#     fill_value=0
# )

# 為避免跨 DataFrame index 對齊問題，
# 用各 Dataset 直接計算覆蓋前面的交叉表
label_rows = []

for name, data in datasets.items():
    counts = data["endorsement_label"].value_counts()
    total = len(data)

    label_rows.append({
        "dataset": name,
        "True": counts.get("True", 0),
        "False": counts.get("False", 0),
        "Unknown": counts.get("Unknown", 0),
        "True_%": round(
            counts.get("True", 0) / total * 100, 2
        ) if total else 0,
        "False_%": round(
            counts.get("False", 0) / total * 100, 2
        ) if total else 0,
        "Unknown_%": round(
            counts.get("Unknown", 0) / total * 100, 2
        ) if total else 0
    })

df_label_summary = pd.DataFrame(label_rows)

display(df_label_summary)


# 3. Category 分布：每位人物在每組只計算一次
category_rows = []

for name, data in datasets.items():
    sampled_persons = (
        persons.loc[
            persons["person_group"].isin(
                data["person_group"].unique()
            )
        ].copy()
    )

    denominator = len(sampled_persons)

    for category in all_categories:
        count = sampled_persons["categories"].apply(
            lambda cats: category in cats
        ).sum()

        category_rows.append({
            "dataset": name,
            "category": category,
            "persons": int(count),
            "person_share_%": round(
                count / denominator * 100, 2
            ) if denominator else 0
        })

df_category_summary = pd.DataFrame(category_rows)

display(
    df_category_summary.pivot(
        index="category",
        columns="dataset",
        values="person_share_%"
    ).round(2)
)


# 4. 粉絲規模分布（人物層級）
follower_rows = []

for name, data in datasets.items():
    sampled_persons = persons.loc[
        persons["person_group"].isin(
            data["person_group"].unique()
        )
    ]

    for group in follower_groups:
        follower_rows.append({
            "dataset": name,
            "follower_group": group,
            "persons": int(
                sampled_persons["follower_group"]
                .eq(group).sum()
            )
        })

df_follower_summary = pd.DataFrame(follower_rows)

display(
    df_follower_summary.pivot(
        index="follower_group",
        columns="dataset",
        values="persons"
    ).fillna(0).astype(int)
)


# 5. 最終 Leakage Check
train_people = set(
    datasets["train"]["person_group"]
)
val_people = set(
    datasets["validation"]["person_group"]
)
test_people = set(
    datasets["test"]["person_group"]
)

assert train_people.isdisjoint(val_people)
assert train_people.isdisjoint(test_people)
assert val_people.isdisjoint(test_people)

assert set(df_pilot["person_group"]).issubset(train_people)

print("\nAll group-disjoint checks: PASSED")


# 6. 儲存統計與研究設定
df_split_summary.to_csv(
    OUTPUT_DIR / "summary_overview.csv",
    index=False, encoding="utf-8-sig"
)

df_label_summary.to_csv(
    OUTPUT_DIR / "summary_labels.csv",
    index=False, encoding="utf-8-sig"
)

df_category_summary.to_csv(
    OUTPUT_DIR / "summary_categories.csv",
    index=False, encoding="utf-8-sig"
)

df_follower_summary.to_csv(
    OUTPUT_DIR / "summary_followers.csv",
    index=False, encoding="utf-8-sig"
)

sampling_config = {
    "split_ratio": TARGET_RATIO.tolist(),
    "split_method": "StratifiedGroupKFold + fold selection",
    "group_key": "normalized person_name",
    "split_seed": int(best["seed"]),
    "split_score": float(best["score"]),
    "split_weights": WEIGHTS,
    "pilot_method": "simple random sampling without replacement",
    "pilot_seed": PILOT_SEED,
    "pilot_confidence": CONFIDENCE,
    "pilot_problem_rate": PROBLEM_RATE,
    "pilot_n": pilot_n,
    "label_source": "JSON Gemini isEndorsement"
}

with open(
    OUTPUT_DIR / "sampling_config.json",
    "w", encoding="utf-8"
) as f:
    json.dump(
        sampling_config, f,
        ensure_ascii=False,
        indent=2
    )

print("\nAll datasets and summaries saved:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)

Building summary tables:   0%|          | 0/5 [00:00<?, ?it/s]

,dataset,posts,post_ratio_%,persons,ig_accounts,true,false,unknown,true_ratio_%
0,full,19102,100.00,194,196,9146,9956,0,47.88
1,train,13408,70.19,138,140,6494,6914,0,48.43
2,validation,1910,10.00,19,19,893,1017,0,46.75
3,test,3784,19.81,37,37,1759,2025,0,46.49
4,pilot,299,1.57,97,98,136,163,0,45.48


,dataset,True,False,Unknown,True_%,False_%,Unknown_%
0,full,9146,9956,0,47.88,52.12,0.0
1,train,6494,6914,0,48.43,51.57,0.0
2,validation,893,1017,0,46.75,53.25,0.0
3,test,1759,2025,0,46.49,53.51,0.0
4,pilot,136,163,0,45.48,54.52,0.0


dataset,full,pilot,test,train,validation
category,,,,,
3C科技,3.09,2.06,5.41,2.17,5.26
家庭母嬰,6.70,9.28,5.41,6.52,10.53
寵物,7.22,8.25,13.51,6.52,0.00
帶貨分潤,4.12,5.15,5.41,3.62,5.26
影視評論,2.58,2.06,2.70,2.17,5.26
旅遊,19.07,18.56,18.92,18.84,21.05
時事討論,1.55,2.06,2.70,1.45,0.00
時尚潮流,12.37,10.31,16.22,12.32,5.26
汽機車,1.55,1.03,0.00,1.45,5.26


dataset,full,pilot,test,train,validation
follower_group,,,,,
Q1,49,22,11,32,6
Q2,48,24,10,35,3
Q3,48,22,7,37,4
Q4,49,29,9,34,6
Unknown,0,0,0,0,0



All group-disjoint checks: PASSED

All datasets and summaries saved:
person_split_mapping.csv
pilot_299.csv
sampling_config.json
summary_categories.csv
summary_followers.csv
summary_labels.csv
summary_overview.csv
test.csv
train.csv
validation.csv
